# Register the champion ensemble

Wraps the winning ensemble (20% LightGBM + 80% N-BEATS) as a single MLflow **pyfunc**
that takes the **raw** `test.csv` and returns weekly-sales predictions, then registers
it in the Model Registry as `Walmart_Champion_Ensemble`.

The wrapper bundles everything it needs (the trained LightGBM, the N-BEATS model, the
sales history, and train/stores/features for building the test features), so it runs
on raw test data without re-deriving anything by hand.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import shutil
import yaml
import warnings
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
import dagshub

from src.features.preprocessing import get_model_ready_data
from src.models.lightgbm_pipeline import create_lightgbm_pipeline
from src.features.nn_preprocessing import build_long_df

warnings.filterwarnings("ignore")
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("Champion_Registry")
SPLIT_DATE = "2012-01-01"
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/12 04:12:15 INFO mlflow.tracking.fluent: Experiment with name 'Champion_Registry' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


## Prepare the artifacts the champion needs

In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")

os.makedirs("champion_artifacts", exist_ok=True)

# 1. Train LightGBM on the full training data with the fixed preprocessing.
with open("configs/lightgbm_config.yaml") as f:
    lgb_params = yaml.safe_load(f)["model"]["params"]
X_train, y_train, _, _, _, preprocessor = get_model_ready_data(train_raw, stores, features, SPLIT_DATE)
lgb_pipe = create_lightgbm_pipeline(preprocessor, lgb_params)
lgb_pipe.fit(X_train, y_train)
joblib.dump(lgb_pipe, "champion_artifacts/lgb.joblib")

# 2. N-BEATS sales history (for its forecast) + copies of the raw files.
history = build_long_df(train_raw, stores, features, fill_gaps=True)[["unique_id", "ds", "y"]]
history.to_parquet("champion_artifacts/history.parquet")
for name in ["train.csv", "stores.csv", "features.csv"]:
    shutil.copy(f"data/{name}", f"champion_artifacts/{name}")
print("artifacts ready")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2342
[LightGBM] [Info] Number of data points in the train set: 294132, number of used features: 19
[LightGBM] [Info] Start training from score 16105.306894
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
artifacts ready


## The champion pyfunc (runs on raw test.csv)

In [4]:
class EnsembleChampion(mlflow.pyfunc.PythonModel):
    NUMERIC = ["Store", "Dept", "Temperature", "Fuel_Price_Delta", "CPI_Delta",
               "Unemployment_Delta", "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4",
               "MarkDown5", "Size", "WeekOfYear", "Is_Black_Friday", "Pre_Christmas", "IsHoliday"]
    CATEGORICAL = ["Type"]
    W_LGB, W_NN = 0.2, 0.8

    def load_context(self, context):
        import joblib, pandas as pd
        from neuralforecast import NeuralForecast
        self.lgb = joblib.load(context.artifacts["lgb"])
        self.nf = NeuralForecast.load(path=context.artifacts["nbeats"])
        self.history = pd.read_parquet(context.artifacts["history"])
        self.train = pd.read_csv(context.artifacts["train"])
        self.stores = pd.read_csv(context.artifacts["stores"])
        self.features = pd.read_csv(context.artifacts["features"])
        self.model_col = self.nf.models[0].alias

    def _test_features(self, test):
        import pandas as pd, numpy as np
        te = test.copy()
        te["Weekly_Sales"] = np.nan
        df = pd.concat([self.train, te], ignore_index=True)
        df = df.merge(self.stores, on="Store", how="left")
        df = df.merge(self.features, on=["Store", "Date", "IsHoliday"], how="left")
        df["Date"] = pd.to_datetime(df["Date"])
        df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
        df["Is_Black_Friday"] = ((df["Date"].dt.month == 11) & (df["Date"].dt.day >= 23) & (df["Date"].dt.day <= 29)).astype(int)
        df["Pre_Christmas"] = (df["WeekOfYear"] == 51).astype(int)
        df["IsHoliday"] = df["IsHoliday"].astype(int)
        df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
        for c in ["Fuel_Price", "CPI", "Unemployment"]:
            df[c + "_Delta"] = df.groupby(["Store", "Dept"])[c].diff().fillna(0)
        return df[df["Weekly_Sales"].isna()].copy()

    def predict(self, context, model_input):
        import pandas as pd, numpy as np
        tr = self._test_features(model_input)
        tr["lgb"] = self.lgb.predict(tr[self.NUMERIC + self.CATEGORICAL])
        tr["unique_id"] = tr["Store"].astype(str) + "_" + tr["Dept"].astype(str)
        tr["ds"] = tr["Date"]
        fcst = self.nf.predict(df=self.history).rename(columns={self.model_col: "nn"})
        tr = tr.merge(fcst[["unique_id", "ds", "nn"]], on=["unique_id", "ds"], how="left")
        tr["nn"] = tr["nn"].fillna(0)
        tr["pred"] = np.clip(self.W_LGB * tr["lgb"] + self.W_NN * tr["nn"], 0, None)
        # Align output to the input row order (by Store/Dept/Date).
        tr["Date"] = tr["Date"].dt.strftime("%Y-%m-%d")
        out = model_input.copy()
        out["Date"] = out["Date"].astype(str)
        out = out.merge(tr[["Store", "Dept", "Date", "pred"]], on=["Store", "Dept", "Date"], how="left")
        return out["pred"].fillna(0).values

D:\Desktop\ML Project\.venv\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## Log and Register the champion

In [5]:
artifacts = {
    "lgb": "champion_artifacts/lgb.joblib",
    "nbeats": "saved_models/nbeats_nf",
    "history": "champion_artifacts/history.parquet",
    "train": "champion_artifacts/train.csv",
    "stores": "champion_artifacts/stores.csv",
    "features": "champion_artifacts/features.csv",
}

with mlflow.start_run(run_name="Register_Champion_Ensemble"):
    mlflow.log_param("weights", "0.2*LightGBM + 0.8*NBEATS")
    mlflow.log_metric("val_WMAE", 2209.78)
    info = mlflow.pyfunc.log_model(
        name="model",
        python_model=EnsembleChampion(),
        artifacts=artifacts,
        pip_requirements=["neuralforecast", "torch", "lightgbm", "scikit-learn",
                          "pandas", "numpy", "joblib", "utilsforecast"],
        registered_model_name="Walmart_Champion_Ensemble",
    )
    print("Registered:", info.model_uri)

2026/07/12 04:13:00 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Successfully registered model 'Walmart_Champion_Ensemble'.
2026/07/12 04:14:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_Champion_Ensemble, version 1
Created version '1' of model 'Walmart_Champion_Ensemble'.


Registered: models:/m-305902d351d1439db5b5dedf58f5759e
🏃 View run Register_Champion_Ensemble at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/11/runs/7cae883fa88c47a7a259ce9caaff1260
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/11


## Quick self-test (predict on raw test)

In [6]:
test_raw = pd.read_csv("data/test.csv")
loaded = mlflow.pyfunc.load_model(info.model_uri)
preds = loaded.predict(test_raw[["Store", "Dept", "Date", "IsHoliday"]])
print("predictions:", len(preds), "| min:", float(np.min(preds)), "| max:", float(np.max(preds)))

2026-07-12 04:14:52,939	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-12 04:14:53,745	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


Predicting DataLoader 0: 100%|██████████| 105/105 [00:01<00:00, 60.28it/s]
predictions: 115064 | min: 0.0 | max: 201084.72191854686
